# Monitoring Notebook

Builds `bda_gold.brand_monitoring` from the dense monthly Gold result.
Includes share-change explanation and alert testing.

## Alert Rule
Alert when Brand#32's share in any category falls by at least 5 percentage points
compared with the previous complete consecutive month.

Boundary months (first and last) are excluded as potentially incomplete.
The production alert query is in `sql/brand_share_alert.sql`.

In [ ]:
spark.sql("USE CATALOG workspace")
dbutils.notebook.run("./03_validation", 0, {"validation_scope": "gold"})


In [ ]:
%sql
-- observed coverage boundaries (assumption: boundary months may be partial)
SELECT
    MIN(order_date) AS observed_min_order_date,
    MAX(order_date) AS observed_max_order_date,
    CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS coverage_min_month,
    CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS coverage_max_month,
    COUNT(DISTINCT CAST(date_trunc('MONTH', order_date) AS DATE)) AS month_count
FROM bda_gold.brand_sales;

In [ ]:
%sql
-- build from dense monthly result, not recomputed from brand_sales
-- boundary months (first and last) excluded as potentially incomplete
CREATE OR REPLACE TABLE bda_gold.brand_monitoring AS
WITH date_bounds AS (
    SELECT
        CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS min_month,
        CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS max_month
    FROM bda_gold.brand_sales
),
with_lag AS (
    SELECT
        m.month_start,
        m.brand,
        m.category,
        m.brand_revenue,
        m.category_revenue,
        m.market_share,
        LAG(m.market_share) OVER (
            PARTITION BY m.brand, m.category ORDER BY m.month_start
        ) AS previous_market_share,
        LAG(m.month_start) OVER (
            PARTITION BY m.brand, m.category ORDER BY m.month_start
        ) AS previous_month_start
    FROM bda_gold.brand_category_monthly m
)
SELECT
    w.month_start,
    w.brand,
    w.category,
    w.brand_revenue,
    w.category_revenue,
    w.market_share,
    w.previous_market_share,
    (w.market_share - w.previous_market_share) * 100 AS share_change_percentage_points,
    CASE
        WHEN w.month_start > (SELECT min_month FROM date_bounds)
         AND w.month_start < (SELECT max_month FROM date_bounds)
        THEN true ELSE false
    END AS is_complete_month,
    CASE
        WHEN w.month_start > (SELECT min_month FROM date_bounds)
         AND w.month_start < (SELECT max_month FROM date_bounds)
         AND ADD_MONTHS(w.month_start, -1) > (SELECT min_month FROM date_bounds)
         AND w.market_share IS NOT NULL
         AND w.previous_market_share IS NOT NULL
         AND w.previous_month_start = ADD_MONTHS(w.month_start, -1)
        THEN true ELSE false
    END AS is_comparable_month
FROM with_lag w;

In [ ]:
%sql
-- Coverage report: brand-category-months with zero revenue
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN brand_revenue = 0 THEN 1 ELSE 0 END) AS zero_revenue_rows,
    SUM(CASE WHEN market_share IS NULL THEN 1 ELSE 0 END) AS null_share_rows,
    SUM(CASE WHEN is_complete_month THEN 1 ELSE 0 END) AS complete_months,
    SUM(CASE WHEN is_comparable_month THEN 1 ELSE 0 END) AS comparable_months,
    COUNT(DISTINCT month_start) AS distinct_months
FROM bda_gold.brand_monitoring;

-- Latest 20 rows for Brand#32
SELECT
    month_start, category,
    ROUND(brand_revenue, 2) AS brand_revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(previous_market_share * 100, 2) AS prev_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_complete_month, is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
ORDER BY month_start DESC, category
LIMIT 20;

In [ ]:
%sql
-- Latest comparable month's Brand#32 shares and changes
SELECT
    month_start, category,
    ROUND(market_share * 100, 2) AS market_share_pct,
    ROUND(share_change_percentage_points, 2) AS share_change_pp,
    is_comparable_month
FROM bda_gold.brand_monitoring
WHERE brand = 'Brand#32'
  AND is_comparable_month
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_monitoring
      WHERE is_comparable_month AND brand = 'Brand#32'
  )
ORDER BY category;

In [ ]:
%sql
-- arithmetic decomposition of share change for comparable brand-category-months
-- b0=prev brand rev, b1=curr brand rev, c0=prev cat rev, c1=curr cat rev, s0=b0/c0
-- brand_effect = 100*(b1-b0)/c1
-- category_effect = -100*s0*(c1-c0)/c1
-- sum = 100*(b1/c1 - b0/c0)
CREATE OR REPLACE TABLE bda_gold.brand_share_change_explanation AS
WITH with_lag AS (
    SELECT
        month_start, brand, category,
        brand_revenue, category_revenue,
        market_share, previous_market_share,
        is_comparable_month,
        LAG(brand_revenue) OVER (PARTITION BY brand, category ORDER BY month_start) AS prev_brand_rev,
        LAG(category_revenue) OVER (PARTITION BY brand, category ORDER BY month_start) AS prev_cat_rev
    FROM bda_gold.brand_monitoring
)
SELECT
    month_start, brand, category,
    brand_revenue, prev_brand_rev,
    category_revenue, prev_cat_rev,
    market_share, previous_market_share,
    100 * (brand_revenue - prev_brand_rev) / category_revenue AS brand_revenue_effect_pp,
    -100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue AS category_revenue_effect_pp,
    100 * (brand_revenue - prev_brand_rev) / category_revenue
        + (-100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue) AS total_effect_pp,
    100 * (market_share - previous_market_share) AS actual_share_change_pp,
    ABS(
        100 * (brand_revenue - prev_brand_rev) / category_revenue
        + (-100 * previous_market_share * (category_revenue - prev_cat_rev) / category_revenue)
        - 100 * (market_share - previous_market_share)
    ) AS identity_error_pp
FROM with_lag
WHERE is_comparable_month;

In [ ]:
%sql
-- Brand#32 latest comparable share-change explanation
SELECT
    month_start, category,
    ROUND(brand_revenue_effect_pp, 4) AS brand_effect_pp,
    ROUND(category_revenue_effect_pp, 4) AS category_effect_pp,
    ROUND(total_effect_pp, 4) AS total_effect_pp,
    ROUND(actual_share_change_pp, 4) AS actual_change_pp,
    ROUND(identity_error_pp, 8) AS identity_error_pp
FROM bda_gold.brand_share_change_explanation
WHERE brand = 'Brand#32'
  AND month_start = (
      SELECT MAX(month_start)
      FROM bda_gold.brand_share_change_explanation
      WHERE brand = 'Brand#32'
  )
ORDER BY category;

In [ ]:
%sql
-- validate decomposition identity across all rows
SELECT
    COUNT(*) AS total_rows,
    MAX(identity_error_pp) AS max_identity_error_pp,
    CASE WHEN MAX(identity_error_pp) < 0.001 THEN 'PASS' ELSE 'FAIL' END AS identity_check
FROM bda_gold.brand_share_change_explanation;

## Alert Demonstration

Synthetic monitoring data tests all alert scenarios.
Real Gold tables are not modified.

In [ ]:
from decimal import Decimal

tests = [
    ("unchanged", "100.00", False, False, 0),
    ("minus 4.99pp", "50.10", False, False, 0),
    ("minus 5pp", "50.00", False, False, 1),
    ("minus 6pp", "40.00", False, False, 1),
    ("plus 6pp", "160.00", False, False, 0),
    ("incomplete", "40.00", True, False, 0),
    ("missing baseline", "40.00", False, True, 0),
]
for label, revenue, incomplete, missing, expected in tests:
    current = Decimal(revenue)
    baseline = "" if missing else "UNION ALL SELECT DATE '1996-02-01', CAST(100 AS DECIMAL(18,2)), CAST(1000 AS DECIMAL(18,2))"
    current_date = "1996-04-01" if incomplete else "1996-03-01"
    result = spark.sql(f"""
        WITH revenues AS (
            SELECT DATE '{current_date}' AS month_start,
                   CAST({current} AS DECIMAL(18,2)) AS brand_revenue,
                   CAST(1000 AS DECIMAL(18,2)) AS category_revenue
            {baseline}
        ), shares AS (
            SELECT *, brand_revenue / NULLIF(category_revenue,0) AS market_share
            FROM revenues
        ), lagged AS (
            SELECT *, LAG(market_share) OVER (ORDER BY month_start) AS previous_share,
                      LAG(month_start) OVER (ORDER BY month_start) AS previous_month
            FROM shares
        )
        SELECT COUNT(*) FROM lagged
        WHERE month_start > DATE '1996-01-01' AND month_start < DATE '1996-04-01'
          AND ADD_MONTHS(month_start,-1) > DATE '1996-01-01'
          AND previous_month=ADD_MONTHS(month_start,-1)
          AND market_share IS NOT NULL AND previous_share IS NOT NULL
          AND (market_share-previous_share)*100 <= -5
    """).first()[0]
    assert result == expected, f"{label}: expected {expected}, got {result}"
    print(f"pass: {label}")


In [ ]:
dbutils.notebook.run("./03_validation", 0, {"validation_scope": "all"})
